# A tour of Starsim

This page provides a (relatively) quick tour of Starsim. It builds up a
model step by step: a simple epidemic, then with age groups, an
intervention, data to fit it to, and multiple runs with uncertainty.

To follow along, install Starsim with `pip install "starsim[jax]"`, then
download this page as a Jupyter notebook from the “Jupyter notebook”
link beside the page’s contents. You can also clone the [Starsim
repository](https://github.com/starsimhub/starsim) and open
`docs/tour.qmd` in Jupyter (with `jupytext` installed, Jupyter opens
`.qmd` files directly).

## An SIR model

Let’s start with the standard SIR model. Everyone is in one of three
compartments: susceptible (S), infectious (I), or recovered (R). The
equations are:

$$
\begin{aligned}
\frac{dS}{dt} &= -\beta \frac{S I}{N} \\
\frac{dI}{dt} &= \beta \frac{S I}{N} - \gamma I \\
\frac{dR}{dt} &= \gamma I
\end{aligned}
$$

where N = S + I + R is the population size. People change state via two
*transitions*:

- **Infection**, from S to I. Each susceptible person is infected at a
  rate of β times the fraction of people who are infectious, so the
  number infected per day is β S I / N. In Starsim, a transition’s rate
  is the rate for each person in its source state (per capita), so the
  rate of infection is written without the S: β I / N, the force of
  infection.
- **Recovery**, from I to R. People stay infectious for 10 days on
  average, so they recover at a rate of γ = 1/10 per day.

In Starsim, the compartments are called *states*, and the transitions
say where people move from, to, and at what rate (after `@`):

In [ ]:
import sciris as sc
import starsim as ss

S, I, R = ss.states('S I R')
infection = S >> I @ 0.3*I        # From S to I, at a rate of 0.3 times the fraction infectious
recovery  = I >> R @ ss.days(10)  # From I to R, after 10 days on average
sir = ss.Disease(infection, recovery)

A state in a rate, like `I` in `0.3*I`, is the fraction of people in it.
A duration, like `ss.days(10)`, is the average time spent in a state
before leaving it. Since the duration is in days, β = 0.3 is per day.
You can check what equations Starsim interpreted from what you gave it:

In [ ]:
sir.equations()

To run the model, put it in a sim and say how long to run it for:

In [ ]:
sim = ss.Sim(sir, dur=ss.days(100))
sim.run()
sim.sir.plot()

The sim says what it ran and some key results: 10,000 people (the
default population) as an ODE, R0 = 3 as expected (=0.3/0.1), and the
number of infections. Since nothing in this model is about individual
people, Starsim runs it as a deterministic compartmental model (an ODE).
Which population keyword you give chooses this: `total_pop=` (or none)
runs compartments, `n_agents=` runs agents, and both run agents that
each stand for several people
([Solvers](user_guide/running_solvers.qmd#how-starsim-chooses)). The
same model can also be run as a stochastic model or as agents, as [the
last section](#uncertainty-and-many-runs) shows.

## Checking the model

Starsim has defaults for pretty much everything (all of which can of
course be overridden). `describe()` lists the whole model as Starsim has
built it: every state and transition, plus any parameters that used
Starsim defaults:

In [ ]:
sim.describe()

## Age groups and contacts

Many diseases spread differently between age groups. `stratify()` splits
each state into substates: here we split into three age groups.
[`ss.contacts()`](./api/data.html#starsim.data.contacts "0") says how
the groups mix with each other: the entry in row *i* and column *j* is
the number of contacts a day that people in *i* have with people in *j*.
β is the rate of infection per contact. This model also has an outcome,
`cases`, representing the infections that are reported.

In [ ]:
S, I, R = ss.states('S I R')
beta = ss.par(0.02, bounds=[0.005, 0.1]) # A parameter that can be fitted, between these bounds
dur_inf = ss.par(ss.days(10))            # The mean infectious period

infection = S >> I @ beta*I
recovery  = I >> R @ dur_inf
cases     = ss.outcome(0.2*infection) # One infection in five is reported
sir = ss.Disease(infection, recovery, cases, init_prev=0.0001)
sir.stratify(age={0: 0.3, 15: 0.6, 65: 0.1}) # Ages 0-14, 15-64, and 65+, with the fraction of people in each

contacts = ss.contacts(
    ages=[0, 15, 65],
    matrix=[
        [18, 2, 1],
        [1, 4, 1],
        [3, 6, 5]] # Reciprocal: e.g. 0.3 × 2 contacts of children with adults = 0.6 × 1 of adults with children
)
pars = dict(networks=contacts, total_pop=1e6, start='2025-01-01', stop='2025-07-01')
sim = ss.Sim(sir, **pars)
sim.run()
sim.results.sir.cases.by('age').plot()

Every state and transition has a result of the same name (`S`, `I`, `R`,
`infection`, `recovery`), and so does each outcome (`cases`).
`by('age')` splits a result by age group, and `select()` picks one
group, e.g. `sim.results.sir.I.select(age='65+')`.

## Interventions

As was drilled into everyone’s heads during the COVID pandemic,
interventions can be pharmaceutical or non-pharmaceutical. Here we
introduce one of each: masks, which cut everyone’s transmission by 40%;
and vaccinating 80% of children under 15 with a vaccine that cuts their
chance of infection by 90%:

In [ ]:
masks = ss.modify(sir.pars.beta, multiply=0.6, start='2025-01-20')
vaccination = ss.vaccinate(efficacy=0.9, who=sir.age < 15, when='2025-01-20', coverage=0.8)

Masks are a *window*, in place from `start=` on (until `stop=`, if
given); vaccination is an *event*, a campaign on the day given by
`when=`.

To compare them, make a sim for each scenario, and run them in parallel
with [`ss.parallel()`](./api/run.html#starsim.run.parallel "0"):

In [ ]:
scenarios = {'No intervention': None, 'Masks': masks, 'Vaccine': vaccination}
sims = {}
for label, intervention in scenarios.items():
    sims[label] = ss.Sim(sir, interventions=intervention, **pars)

msim = ss.parallel(sims)
msim.plot('sir.infection')

`msim.summarize()` gives each result over the whole run (the total of a
flow, such as infections, and the final value of a state) for each
scenario; here, the infections:

In [ ]:
msim.summarize('sir.infection')

Masks prevent about a third of infections, and vaccinating children
nearly half.

## Data and fitting

A model is fitted by finding the values of its parameters that make its
results match data. We can use our model to create synthetic data and
then fit to it. Here we set β = 0.02, and draw weekly counts of cases
from it, with random noise as in real reporting. Real data never come
from exactly the model being fitted, so here the data come from a
slightly different one, in which the infectious period varies less from
person to person (an Erlang distribution with 3 stages, rather than the
exponential of the model). The noise is drawn with the true sim’s own
random seed (`rand_seed=4`), and we use a stochastic solver
(tau-leaping) rather than an ODE.

In [ ]:
truth = ss.Sim(sir, pars=dict(beta=0.02, dur_inf=ss.erlang(k=3, mean=ss.days(10))), rand_seed=4, solver='tau', **pars)
truth.run()
data = truth.to_synthetic_data('cases', resample='week', noise=ss.nbinom(n=5))
data.head()

A forecast is made from the data so far. Here, the first four weeks of
cases, up to 1 February, are the data, and the sim stops when they do.
Data is matched to results by name: the column `cases` is compared with
the result `cases`, and since the data is weekly, each row is compared
with the total of the model’s cases over its week. `sim.fit()` then fits
every parameter that has bounds (here, β) to every result that has data:

In [ ]:
early = data[data.date <= '2025-01-22']
sim = ss.Sim(sir, data=early, **sc.mergedicts(pars, dict(stop='2025-02-01')))
fit = sim.fit()
fit.plot()

The plot shows the model’s cases with the starting value of β (before)
and the fitted one (after), against the data. `fit.pars` has the values
found:

In [ ]:
fit.pars

The fitted β is higher than the actual 0.02, since the fitted model’s
exponential infectious period gives a slower growth than the Erlang
distribution.

`fit.project()` runs the fitted model on, here to 1 July, for the
forecast; given all the data, its plot shows the cases that were still
to come:

In [ ]:
forecast = fit.project(stop='2025-07-01', data=data)
forecast.plot('sir.cases')

From the first four weeks alone, the forecast gets the timing and size
of the peak about right.

## Uncertainty and many runs

The ODE gives the deterministic (mean-field) course of the epidemic, but
real epidemics also include stochastic variability, especially when
there are a small number of cases. To see this, we run the fitted model
in a population of 20,000 as a stochastic model, in which the number of
people who move between states each day is random. We do this by setting
`solver='binomial'` (short for [chain
binomial](https://en.wikipedia.org/wiki/Reed%E2%80%93Frost_model)), and
`n_runs=40` runs it 40 times, each with its own random seed:

In [ ]:
sim = ss.Sim(sir, pars=fit.pars, solver='binomial', **sc.mergedicts(pars, dict(total_pop=20_000)))
msim = ss.parallel(sim, n_runs=40)
msim.plot(['sir.I', 'sir.cases'], lines=True)

The plot shows each of the runs (omit `lines=True` to show the median
and 95% range instead). The results are available as a dataframe via
`msim.summarize()`:

In [ ]:
msim.summarize()

Most runs infect nearly the same number of people (a few die out early,
which is why the low end of the range is near zero), but the epidemics
vary considerably in terms of peak timing.

## Next steps

- The [tutorials](tutorials.qmd) cover each of these steps in more
  depth, one at a time, with a model of a measles outbreak.
- The [user guide](user_guide.qmd) covers each topic in full, including
  networks, demographics, custom modules, and solvers.
- The [gallery](gallery.qmd) has complete models to start from.
- If you have a model in another framework, the [conversion
  guides](user_guide/library_import.qmd) show how to bring it into
  Starsim.